# No-Use Eval Analysis

This notebook visualizes `evaluate_no_use_worlds.py` outputs.

Recommended input files:
- `data/no_use/no_use_eval_results_scope.csv`
- `data/no_use/no_use_eval_results_temporal_scope.csv`
- `data/no_use/no_use_eval_summary_scope.json`
- `data/no_use/no_use_eval_summary_temporal_scope.json`


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)


In [ ]:
source_dir = Path("/mnt/yao_data/proj_2026_agent/PersonaMem-main")

SCOPE_CSV = source_dir / 'data/no_use/no_use_eval_results_scope.csv'
TEMPORAL_CSV = source_dir / 'data/no_use/no_use_eval_results_temporal_scope.csv'
SCOPE_SUMMARY = source_dir / 'data/no_use/no_use_eval_summary_scope.json'
TEMPORAL_SUMMARY = source_dir / 'data/no_use/no_use_eval_summary_temporal_scope.json'

assert SCOPE_CSV.exists() or TEMPORAL_CSV.exists(), 'No input CSV found.'


In [ ]:
def load_one(csv_path: Path, mode_name: str):
    if not csv_path.exists():
        return pd.DataFrame()
    df = pd.read_csv(csv_path)
    df['dataset_mode'] = mode_name
    return df

df_scope = load_one(SCOPE_CSV, 'scope')
df_temp = load_one(TEMPORAL_CSV, 'temporal_scope')
df = pd.concat([df_scope, df_temp], ignore_index=True) if (len(df_scope) + len(df_temp)) else pd.DataFrame()

if df.empty:
    raise ValueError('Loaded dataframe is empty. Check CSV paths.')

# Normalize score column
if 'score' not in df.columns and 'policy_score' in df.columns:
    df['score'] = df['policy_score']

for c in ['score', 'policy_score', 'world_score', 'gap_reveal_on', 'gap_on_ask', 'gap_reveal_ask', 'gap_on_off', 'gap_off_ask']:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

df['correct'] = df['score'].fillna(0).astype(float)
df.head(3)


In [ ]:
print('rows:', len(df))
display(df[['dataset_mode','world','eval_bucket','qa_type','topic','ask_period','target_period']].describe(include='all').T)

for p in [SCOPE_SUMMARY, TEMPORAL_SUMMARY]:
    if p.exists():
        print('\n==', p, '==')
        s = json.loads(p.read_text(encoding='utf-8'))
        print(json.dumps(s.get('comparison', {}), indent=2))


In [ ]:
# Overall world-level scores
overall = (
    df.groupby(['dataset_mode','world','eval_bucket'], dropna=False)['correct']
      .agg(['mean','count'])
      .reset_index()
      .rename(columns={'mean':'success_rate','count':'n'})
)
display(overall.sort_values(['dataset_mode','eval_bucket','world']))


In [ ]:
plt.figure(figsize=(9,4))
sns.barplot(data=overall, x='eval_bucket', y='success_rate', hue='world')
plt.ylim(0,1)
plt.title('Overall Success by Bucket and World')
plt.show()


In [ ]:
# By qa_type
by_qa = (
    df.groupby(['dataset_mode','world','qa_type'], dropna=False)['correct']
      .agg(['mean','count'])
      .reset_index()
      .rename(columns={'mean':'success_rate','count':'n'})
)
display(by_qa.sort_values(['dataset_mode','qa_type','world']))

plt.figure(figsize=(12,4))
sns.barplot(data=by_qa, x='qa_type', y='success_rate', hue='world')
plt.xticks(rotation=25, ha='right')
plt.ylim(0,1)
plt.title('Success by QA Type')
plt.show()


In [ ]:
# By topic
by_topic = (
    df.groupby(['dataset_mode','world','topic'], dropna=False)['correct']
      .agg(['mean','count'])
      .reset_index()
      .rename(columns={'mean':'success_rate','count':'n'})
)
display(by_topic.sort_values(['dataset_mode','topic','world']))

g = sns.catplot(
    data=by_topic, kind='bar', x='topic', y='success_rate', hue='world', col='dataset_mode',
    height=4, aspect=1.4
)
for ax in g.axes.flatten():
    ax.set_ylim(0,1)
    for tick in ax.get_xticklabels():
        tick.set_rotation(25)
        tick.set_ha('right')
plt.show()


In [ ]:
# ask_period x target_period heatmaps (no_use bucket)
sub = df[df['eval_bucket'] == 'no_use'].copy()
if len(sub) == 0:
    print('No no_use bucket rows found.')
else:
    for dm in sorted(sub['dataset_mode'].dropna().unique()):
        cur_dm = sub[sub['dataset_mode'] == dm]
        worlds = sorted(cur_dm['world'].dropna().unique())
        fig, axes = plt.subplots(1, len(worlds), figsize=(6*len(worlds), 4), squeeze=False)
        for i, w in enumerate(worlds):
            cur = cur_dm[cur_dm['world'] == w]
            pivot = cur.pivot_table(index='target_period', columns='ask_period', values='correct', aggfunc='mean')
            sns.heatmap(pivot, annot=True, fmt='.2f', vmin=0, vmax=1, cmap='YlGnBu', ax=axes[0, i])
            axes[0, i].set_title(f'{dm} | {w} | no_use')
        plt.tight_layout()
        plt.show()


In [ ]:
# Gap effects
gap_cols = ['gap_on_ask','gap_reveal_on','gap_reveal_ask','gap_on_off','gap_off_ask']
for gc in gap_cols:
    if gc not in df.columns:
        continue
    cur = df[df[gc].notna()].copy()
    if cur.empty:
        continue
    agg = (
        cur.groupby(['dataset_mode','world','eval_bucket',gc])['correct']
           .agg(['mean','count'])
           .reset_index()
           .rename(columns={'mean':'success_rate','count':'n'})
    )
    display(agg.sort_values(['dataset_mode','eval_bucket',gc,'world']).head(30))
    g = sns.relplot(
        data=agg, kind='line', x=gc, y='success_rate', hue='world', col='eval_bucket', row='dataset_mode',
        marker='o', height=3.4, aspect=1.4
    )
    for ax in g.axes.flatten():
        if ax is not None:
            ax.set_ylim(0,1)
    g.fig.suptitle(f'Success vs {gc}', y=1.02)
    plt.show()


In [ ]:
# World-to-world deltas on same strata
group_cols = ['dataset_mode','eval_bucket','qa_type','topic','ask_period','target_period']
tmp = (
    df.groupby(group_cols + ['world'])['correct']
      .mean()
      .reset_index()
)
wide = tmp.pivot_table(index=group_cols, columns='world', values='correct').reset_index()
if 'baseline' in wide.columns and 'no_use' in wide.columns:
    wide['delta_no_use_minus_baseline'] = wide['no_use'] - wide['baseline']
display(wide.sort_values('delta_no_use_minus_baseline').head(20))
display(wide.sort_values('delta_no_use_minus_baseline', ascending=False).head(20))
